# Train Unconditional DDPM Prior on PDE Snapshots

A100 40GB preset, bf16 mixed precision, full dataset loaded into RAM. The notebook trains an unconditional diffusion prior first, saves generated previews every epoch, validates every epoch, and downloads the best checkpoint whenever validation loss improves.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

## Clone repo and install dependencies

In [ ]:
import subprocess

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "dl_final_project_training"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Dataset source

`DATA_SOURCE` can be one of:
- public Yandex Disk file/folder link. For folder links the loader downloads `.npz` and `.json` files individually and reuses already cached files;
- local Colab path to a `.zip`, `.npz`, or folder with `.npz` chunks;
- mounted Google Drive path.

In [ ]:
DATA_SOURCE = "https://disk.yandex.ru/d/rrjDGzzX5cfFnA"

# Example local paths after manual upload or Drive mount:
# DATA_SOURCE = "/content/kolmogorov_velocity_256_to_64_100k_splits.zip"
# DATA_SOURCE = "/content/drive/MyDrive/datasets/kolmogorov_velocity_256_to_64_100k_splits.zip"

assert DATA_SOURCE, "Set DATA_SOURCE first"

## Configure training

This cell only creates the config. It does not download data and does not start training. Defaults are intended for `64x64` two-channel Kolmogorov velocity snapshots. If CUDA memory is tight, lower `batch_size` to `128` or `base_channels` to `96`.

In [ ]:
from diffusion_training import TrainConfig, prepare_dataset, train_diffusion_model

config = TrainConfig(
    data_source=DATA_SOURCE,
    output_dir="runs",
    cache_dir="data/download_cache",
    stats_cache_path="data/download_cache/kolmogorov_velocity_256_to_64_train_stats.json",
    force_recompute_stats=False,
    dataset_tag="kolmogorov_velocity_256_to_64",
    epochs=100,
    batch_size=256,
    val_batch_size=256,
    num_workers=4,
    lr=2.0e-4,
    weight_decay=1.0e-4,
    grad_accum_steps=1,
    timesteps=1000,
    beta_schedule="cosine",
    sample_steps=250,
    sample_count=32,
    sample_every_epochs=1,
    display_samples_in_notebook=True,
    use_ema_for_validation=False,
    use_ema_for_sampling=False,
    base_channels=128,
    channel_mults="1,2,4,4",
    num_res_blocks=2,
    attention_resolutions="16",
    ema_decay=0.9999,
    precision="bf16",
    compile_model=False,
    save_last_every_epochs=1,
    download_best_in_colab=False,
)

config


## Download and load dataset

Run this cell to download/check cached Yandex Disk files, load train/val into RAM, compute or reuse `mean/std`, and normalize the arrays. Training does not start here.

In [ ]:
dataset = prepare_dataset(config)
print("Train samples:", len(dataset.train))
print("Val samples:", len(dataset.val))
print("Stats cache:", dataset.stats.stats_cache_path)


## Start training

Run this cell after the dataset cell. This is the only cell that starts model training. After each epoch it saves unconditional samples and displays the latest sample grid directly below the cell output.

In [ ]:
best_checkpoint = train_diffusion_model(config, dataset=dataset)
print("Best checkpoint:", best_checkpoint)


## Inspect training history

In [ ]:
import json
from pathlib import Path

run_dir = Path(best_checkpoint).parents[1]
history = json.loads((run_dir / "history.json").read_text())["history"]
print("Run dir:", run_dir)
print(json.dumps(history[-5:], indent=2))

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_loss = [row["train_loss"] for row in history]
val_loss = [row["val_loss"] for row in history]

plt.figure(figsize=(7, 4))
plt.plot(epochs, train_loss, label="train")
plt.plot(epochs, val_loss, label="val")
plt.xlabel("epoch")
plt.ylabel("DDPM eps MSE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

## View latest unconditional samples

In [ ]:
from IPython.display import display
from PIL import Image

sample_files = sorted((run_dir / "samples").glob("*.png"))
print("Latest sample:", sample_files[-1])
display(Image.open(sample_files[-1]))

## Archive run artifacts

In [ ]:
from google.colab import files

archive_path = f"{run_dir.name}.zip"
!zip -q -r {archive_path} {run_dir}
files.download(archive_path)